In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import pickle
from tensorflow.keras.models import load_model

In [2]:
# load the trained model, scaler pickle and onehot
model=load_model('churn_model.h5')

with open('label_encoder.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)
with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)
with open('one_hot_encoder.pkl','rb') as file:
    onehot_encoder=pickle.load(file)
    

In [3]:
df = pd.read_csv('Churn_Modelling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
input_data = {
   'credit_score': 600,
   'Geography': 'France',
   'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1, 
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
    
    
}

In [5]:
geo_encoder = onehot_encoder.transform([[input_data['Geography']]]).toarray()
geo_encoder_df = pd.DataFrame(geo_encoder, columns=onehot_encoder.get_feature_names_out(['Geography']))
geo_encoder_df.head()

c:\tareq\Data-science\project\end-to-end-deep-learning-project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [6]:
gender_encoded = label_encoder_gender.transform([input_data['Gender']])[0]  
input_data['Gender']=gender_encoded

In [11]:
## concatenation one hot encoded 
input_series = pd.Series(input_data)  # Convert dict to Series
input_without_geo = input_series.drop("Geography")  # Drop Geography column
input_df = pd.concat([input_without_geo, geo_encoder_df.iloc[0]], axis=0)  # Merge with one-hot encoded geography
input_df = pd.DataFrame([input_df])
input_df.head()

,credit_score,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [15]:
## Scaling the input data

# Rename column to match scaler training data
input_df = input_df.rename(columns={'credit_score': 'CreditScore'})

input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])